##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: Question Answering using LangChain and Chroma

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/langchain/Gemini_LangChain_QA_Chroma_WebLoad.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

> **Note:** This notebook requires paid tier rate limits to run properly. (cf. [pricing](https://ai.google.dev/pricing) for more details).


## Overview

[Gemini](https://ai.google.dev/models/gemini) is a family of generative AI models that lets developers generate content and solve problems. These models are designed and trained to handle both text and images as input.

[LangChain](https://www.langchain.com/) is a data framework designed to make integration of Large Language Models (LLM) like Gemini easier for applications.

[Chroma](https://docs.trychroma.com/) is an open-source embedding database focused on simplicity and developer productivity. Chroma allows users to store embeddings and their metadata, embed documents and queries, and search the embeddings quickly.

In this notebook, you'll learn how to create an application that answers questions using data from a website with the help of Gemini, LangChain, and Chroma.

## Setup

First, you must install the packages and set the necessary environment variables.

### Installation

Install LangChain's Python library, `langchain` and LangChain's integration package for Gemini, `langchain-google-genai`. Next, install Chroma's Python client SDK, `chromadb`.

In [ ]:
!pip install langchain langchain-core langchain-community langchain-google-genai chromadb bs4 jq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.8/773.8 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/6

In [ ]:
# %pip install --quiet langchain-core==0.1.23
# %pip install --quiet langchain==0.1.1
# %pip install --quiet langchain-google-genai==0.0.6
# %pip install --quiet -U langchain-community==0.0.20
# %pip install --quiet chromadb
# %pip install --quiet bs4

# %pip install --quiet langchain-core==0.1.23
# %pip install --quiet langchain==0.1.1
# %pip install --quiet langchain-google-genai==0.0.6
# %pip install --quiet -U langchain-community==0.0.20
# %pip install --quiet chromadb
# %pip install --quiet bs4

In [ ]:
#from langchain import hub
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate, format_document
from langchain_community.document_loaders import WebBaseLoader, JSONLoader
from langchain_community.vectorstores import Chroma

/tmp/ipykernel_16773/3676758637.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader, JSONLoader


## Configure your API key

To run the following cell, your API key must be stored in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) for an example.


In [ ]:
import os
from google.colab import userdata
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

## Basic steps
LLMs are trained offline on a large corpus of public data. Hence they cannot answer questions based on custom or private data accurately without additional context.

If you want to make use of LLMs to answer questions based on private data, you have to provide the relevant documents as context alongside your prompt. This approach is called Retrieval Augmented Generation (RAG).

You will use this approach to create a question-answering assistant using the Gemini text model integrated through LangChain. The assistant is expected to answer questions about the Gemini model. To make this possible you will add more context to the assistant using data from a website.

In this tutorial, you'll implement the two main components in an RAG-based architecture:

1. Retriever

    Based on the user's query, the retriever retrieves relevant snippets that add context from the document. In this tutorial, the document is the website data.
    The relevant snippets are passed as context to the next stage - "Generator".

2. Generator

    The relevant snippets from the website data are passed to the LLM along with the user's query to generate accurate answers.

You'll learn more about these stages in the upcoming sections while implementing the application.

## Retriever

In this stage, you will perform the following steps:

1. Read and parse the website data using LangChain.

2. Create embeddings of the website data.

    Embeddings are numerical representations (vectors) of text. Hence, text with similar meaning will have similar embedding vectors. You'll make use of Gemini's embedding model to create the embedding vectors of the website data.

3. Store the embeddings in Chroma's vector store.
    
    Chroma is a vector database. The Chroma vector store helps in the efficient retrieval of similar vectors. Thus, for adding context to the prompt for the LLM, relevant embeddings of the text matching the user's question can be retrieved easily using Chroma.

4. Create a Retriever from the Chroma vector store.

    The retriever will be used to pass relevant website embeddings to the LLM along with user queries.

### Read and parse the website data

LangChain provides a wide variety of document loaders. To read the website data as a document, you will use the `WebBaseLoader` from LangChain.

To know more about how to read and parse input data from different sources using the document loaders of LangChain, read LangChain's [document loaders guide](https://python.langchain.com/docs/integrations/document_loaders).

In [ ]:
# loader = WebBaseLoader("https://blog.google/technology/ai/google-gemini-ai/")
# docs = loader.load()

In [ ]:
loader = JSONLoader(file_path='/content/recettes_details.json',jq_schema='.[]', text_content=False)
docs = loader.load()

FileNotFoundError: [Errno 2] No such file or directory: '/content/recettes_details.json'

If you only want to select a specific portion of the website data to add context to the prompt, you can use regex, text slicing, or text splitting.

In this example, you'll use Python's `split()` function to extract the required portion of the text. The extracted text should be converted back to LangChain's `Document` format.

In [ ]:
# # Extract the text from the website data document
# text_content = docs[0].page_content

# # The text content between the substrings "code, audio, image and video." to
# # "Cloud TPU v5p" is relevant for this tutorial. You can use Python's `split()`
# # to select the required content.
# text_content_1 = text_content.split("code, audio, image and video.",1)[1]
# final_text = text_content_1.split("Cloud TPU v5p",1)[0]

# # Convert the text to LangChain's `Document` format
# docs = [Document(page_content=final_text, metadata={"source": "local"})]

### Initialize Gemini's embedding model

To create the embeddings from the website data, you'll use Gemini's embedding model, **gemini-embedding-001** which supports creating text embeddings.

To use this embedding model, you have to import `GoogleGenerativeAIEmbeddings` from LangChain. To know more about the embedding model, read Google AI's [language documentation](https://ai.google.dev/models/gemini).

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

gemini_embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

### Store the data using Chroma

To create a Chroma vector database from the website data, you will use the `from_documents` function of `Chroma`. Under the hood, this function creates embeddings from the documents created by the document loader of LangChain using any specified embedding model and stores them in a Chroma vector database.  

You have to specify the `docs` you created from the website data using LangChain's `WebBasedLoader` and the `gemini_embeddings` as the embedding model when invoking the `from_documents` function to create the vector database from the website data. You can also specify a directory in the `persist_directory` argument to store the vector store on the disk. If you don't specify a directory, the data will be ephemeral in-memory.


In [ ]:
import html
import time
from langchain_community.vectorstores import Chroma

for doc in docs:
    doc.page_content = html.unescape(doc.page_content)

ids = [f"{doc.metadata.get('source', 'recette')}_{doc.metadata.get('seq_num', idx)}" for idx, doc in enumerate(docs)]

vectorstore = Chroma(embedding_function=gemini_embeddings, persist_directory="./chroma_db")

batch_size = 10

# Récupère les IDs déjà indexés pour reprendre où on s'est arrêté
deja_indexes = set(vectorstore._collection.get()["ids"])
print(f"Déjà indexés : {len(deja_indexes)} documents")

print(f"Indexation de {len(docs)} documents...")

for i in range(0, len(docs), batch_size):
    batch_docs = docs[i : i + batch_size]
    batch_ids  = ids[i : i + batch_size]

    # Skip si tous les docs du lot sont déjà dans Chroma
    if all(bid in deja_indexes for bid in batch_ids):
        # print(f"Lot {i//batch_size + 1} déjà indexé, on passe")
        continue

    try:
        vectorstore.add_documents(batch_docs, ids=batch_ids)
        print(f"Lot {i//batch_size + 1} ajouté ({min(i + batch_size, len(docs))}/{len(docs)})")
        if i + batch_size < len(docs):
            time.sleep(12)
    except Exception as e:
        print(f"⚠️ Quota atteint au lot {i//batch_size + 1} — relance la cellule dans 1 min, elle reprendra ici")
        break

print("Terminé !")

NameError: name 'docs' is not defined

In [ ]:
import shutil

# Zippe le dossier chroma_db
shutil.make_archive("chroma_db_backup", "zip", ".", "chroma_db")
print("✅ chroma_db_backup.zip créé")

# Télécharge sur ton PC
from google.colab import files
files.download("chroma_db_backup.zip")

FileNotFoundError: [Errno 2] No such file or directory: 'chroma_db'

In [ ]:
import shutil
from google.colab import files

# Upload le zip
uploaded = files.upload()  # sélectionne chroma_db_backup.zip

# Dézippe
shutil.unpack_archive("chroma_db_backup.zip", ".")
print("✅ chroma_db restauré")

Saving chroma_db_backup.zip to chroma_db_backup.zip
✅ chroma_db restauré


### Create a retriever using Chroma

You'll now create a retriever that can retrieve website data embeddings from the newly created Chroma vector store. This retriever can be later used to pass embeddings that provide more context to the LLM for answering user's queries.


To load the vector store that you previously stored in the disk, you can specify the name of the directory that contains the vector store in `persist_directory` and the embedding model in the `embedding_function` arguments of Chroma's initializer.

You can then invoke the `as_retriever` function of `Chroma` on the vector store to create a retriever.

In [ ]:
# Load from disk à ne pas lancer !
vectorstore_disk = Chroma(
                        persist_directory="./chroma_db",       # Directory of db
                        embedding_function=gemini_embeddings   # Embedding model
                   )
# Get the Retriever interface for the store to use later.
# When an unstructured query is given to a retriever it will return documents.
# Read more about retrievers in the following link.
# https://python.langchain.com/docs/modules/data_connection/retrievers/
#
# Since only 1 document is stored in the Chroma vector store, search_kwargs `k`
# is set to 1 to decrease the `k` value of chroma's similarity search from 4 to
# 1. If you don't pass this value, you will get a warning.
retriever = vectorstore_disk.as_retriever(search_kwargs={"k": 6})

# Check if the retriever is working by trying to fetch the relevant docs related
resultats = retriever.invoke("pizza")
print(f"Nombre de documents trouvés : {len(resultats)}")

/tmp/ipykernel_16773/3218529709.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore_disk = Chroma(


Nombre de documents trouvés : 6


In [ ]:
# Méthode avec seuil relatif à ne plus lancer !
#Load from disk
vectorstore_disk = Chroma(
                        persist_directory="./chroma_db",       # Directory of db
                        embedding_function=gemini_embeddings   # Embedding model
                   )
# Get the Retriever interface for the store to use later.
# When an unstructured query is given to a retriever it will return documents.
# Read more about retrievers in the following link.
# https://python.langchain.com/docs/modules/data_connection/retrievers/
#
# Since only 1 document is stored in the Chroma vector store, search_kwargs `k`
# is set to 1 to decrease the `k` value of chroma's similarity search from 4 to
# 1. If you don't pass this value, you will get a warning.
# Nombre maximum de documents récupérés

K = 6
SEUIL_RELATIF = 0.05

def retrieve_documents(query, afficher_scores=False):
    docs_scores = vectorstore_disk.similarity_search_with_relevance_scores(
        query,
        k=K
    )

    if not docs_scores:
        return []

    meilleur_score = docs_scores[0][1]
    seuil_min = meilleur_score - SEUIL_RELATIF

    docs_filtres = [
        doc for doc, score in docs_scores
        if score >= seuil_min
    ]

    if afficher_scores:
        print(f"Meilleur score : {meilleur_score:.4f}")
        print(f"Seuil minimal : {seuil_min:.4f}")

        for i, (doc, score) in enumerate(docs_scores, start=1):
            statut = "GARDÉ" if doc in docs_filtres else "REJETÉ"
            print(f"\nDocument {i} - {statut}")
            print(f"Score : {score:.4f}")
            print(doc.page_content[:200])
            print("-" * 50)

    return docs_filtres
# Check if the retriever is working by trying to fetch the relevant docs related
resultats = retrieve_documents(
    "pizza maison facile a faire en 30 min",
    afficher_scores=True
)

print(f"Nombre de documents gardés : {len(resultats)}")

Meilleur score : 0.6711
Seuil minimal : 0.6211

Document 1 - GARDÉ
Score : 0.6711
{"url": "https://www.750g.com/pizza-maison-r63439.htm", "titre": "Pizza maison", "difficulte": "Facile", "temps_preparation": "30min", "temps_cuisson": "15min", "nb_personnes": 4, "type_quantite": "pe
--------------------------------------------------

Document 2 - GARDÉ
Score : 0.6348
{"url": "https://www.750g.com/pate-a-la-pizza-r97250.htm", "titre": "P\u00e2te \u00e0 pizza maison et rapide", "difficulte": "Facile", "temps_preparation": "20min", "temps_cuisson": null, "nb_personne
--------------------------------------------------

Document 3 - GARDÉ
Score : 0.6314
{"url": "https://www.750g.com/pate-a-pizza-au-monsieur-cuisine-r208857.htm", "titre": "P\u00e2te \u00e0 pizza au Monsieur Cuisine", "difficulte": "Tr\u00e8s facile", "temps_preparation": "5min", "temp
--------------------------------------------------

Document 4 - GARDÉ
Score : 0.6286
{"url": "https://www.750g.com/pizza-margherita-r200618.ht

In [ ]:
# Méthode avec CRAG leger !
import re
#Load from disk
vectorstore_disk = Chroma(
                        persist_directory="./chroma_db",       # Directory of db
                        embedding_function=gemini_embeddings   # Embedding model
                   )
# Get the Retriever interface for the store to use later.
# When an unstructured query is given to a retriever it will return documents.
# Read more about retrievers in the following link.
# https://python.langchain.com/docs/modules/data_connection/retrievers/
#
# Since only 1 document is stored in the Chroma vector store, search_kwargs `k`
# is set to 1 to decrease the `k` value of chroma's similarity search from 4 to
# 1. If you don't pass this value, you will get a warning.
# Nombre maximum de documents récupérés

K = 8
MAX_DOCS_GARDES = 3

def retrieve_documents_crag(query, afficher_scores=False):

    # Étape 1 : récupération vectorielle
    docs_scores = vectorstore_disk.similarity_search_with_relevance_scores(
        query,
        k=K
    )

    if not docs_scores:
        return []

    docs = [doc for doc, score in docs_scores]

    # Étape 2 : construction du prompt CRAG
    docs_text = ""

    for i, doc in enumerate(docs, start=1):
        docs_text += f"\nDOCUMENT {i}:\n{doc.page_content[:1200]}\n"

    filter_prompt = f"""
Tu es un évaluateur de pertinence pour un système RAG de recettes.

Question utilisateur :
{query}

Documents candidats :
{docs_text}

Ta tâche :
- garde uniquement les documents réellement utiles ;
- rejette les documents hors sujet ;
- respecte les contraintes éventuelles (sans four, végétarien, rapide, etc.) ;
- garde au maximum {MAX_DOCS_GARDES} documents ;
- réponds uniquement avec les numéros des documents à conserver.

Exemples :
1,2
3
AUCUN
"""

    # Étape 3 : filtrage LLM
    llm_response = llm.invoke(filter_prompt).content

    if isinstance(llm_response, list):
        llm_response = " ".join(str(x) for x in llm_response)

    llm_response = str(llm_response)

    print("\n=== Décision du LLM ===")
    print(llm_response)

    # Étape 4 : extraction des numéros
    indices = []

    if "AUCUN" not in llm_response.upper():
        for n in re.findall(r"\d+", llm_response):
            idx = int(n)
            if 1 <= idx <= len(docs):
                indices.append(idx)

    indices = indices[:MAX_DOCS_GARDES]

    docs_filtres = [docs[i-1] for i in indices]

    # Affichage détaillé
    if afficher_scores:
        print("\n=== Résultats retrieval + CRAG ===")

        for i, (doc, score) in enumerate(docs_scores, start=1):
            statut = "GARDÉ" if i in indices else "REJETÉ"

            print(f"\nDocument {i} - {statut}")
            print(f"Score : {score:.4f}")
            print(doc.page_content[:200])
            print("-" * 50)

    return docs_filtres

In [ ]:
resultats = retrieve_documents_crag(
    "je veux faire une quiche pour 10 personnes et j'ai des tomates",
    afficher_scores=True
)

print(f"\nNombre de documents gardés : {len(resultats)}")


=== Décision du LLM ===
2,5,8

=== Résultats retrieval + CRAG ===

Document 1 - REJETÉ
Score : 0.6311
{"url": "https://www.750g.com/quiche-facile-au-thon-et-a-la-tomate-r205541.htm", "titre": "Quiche au thon et \u00e0 la tomate facile", "difficulte": "Facile", "temps_preparation": "20min", "temps_cuis
--------------------------------------------------

Document 2 - GARDÉ
Score : 0.6215
{"url": "https://www.750g.com/tarte-au-chevre-tomates-et-courgettes-r6354.htm", "titre": "Tarte au ch\u00e8vre, tomates et courgettes", "difficulte": "Tr\u00e8s facile", "temps_preparation": "20min", 
--------------------------------------------------

Document 3 - REJETÉ
Score : 0.6199
{"url": "https://www.750g.com/quiche-aux-courgettes-tomates-sechees-et-chevre-r75391.htm", "titre": "Quiche aux courgettes, ch\u00e8vre et tomates s\u00e9ch\u00e9es", "difficulte": "Facile", "temps_pr
--------------------------------------------------

Document 4 - REJETÉ
Score : 0.6185
{"url": "https://www.750g.com/tart

## Generator

The Generator prompts the LLM for an answer when the user asks a question. The retriever you created in the previous stage from the Chroma vector store will be used to pass relevant embeddings from the website data to the LLM to provide more context to the user's query.

You'll perform the following steps in this stage:

1. Chain together the following:
    * A prompt for extracting the relevant embeddings using the retriever.
    * A prompt for answering any question using LangChain.
    * An LLM model from Gemini for prompting.
    
2. Run the created chain with a question as input to prompt the model for an answer.


### Initialize Gemini

You must import `ChatGoogleGenerativeAI` from LangChain to initialize your model.
 In this example, you will use **gemini-2.0-flash**, as it supports text summarization. To know more about the text model, read Google AI's [language documentation](https://ai.google.dev/models/gemini).

You can configure the model parameters such as ***temperature*** or ***top_p***,  by passing the appropriate values when initializing the `ChatGoogleGenerativeAI` LLM.  To learn more about the parameters and their uses, read Google AI's [concepts guide](https://ai.google.dev/docs/concepts#model_parameters).

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# To configure model parameters use the `generation_config` parameter.
# eg. generation_config = {"temperature": 0.7, "topP": 0.8, "topK": 40}
# If you only want to set a custom temperature for the model use the
# "temperature" parameter directly.

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1,
)

### Create prompt templates

You'll use LangChain's [PromptTemplate](https://python.langchain.com/docs/how_to/#prompt-templates) to generate prompts to the LLM for answering questions.

In the `llm_prompt`, the variable `question` will be replaced later by the input question, and the variable `context` will be replaced by the relevant text from the website retrieved from the Chroma vector store.

In [ ]:
# Prompt template to query Gemini
llm_prompt_template = """Tu es RAGoût, assistant culinaire chaleureux.

Tu réponds uniquement à partir des recettes du contexte.
N’invente jamais de recette ni d’ingrédient.

SÉLECTION :
- Identifie si l’utilisateur cherche de l’inspiration ou une recette précise.
- Si plusieurs options sont possibles, privilégie la diversité des suggestions.
- Si aucune recette ne correspond bien, dis-le et propose les plus proches.

QUANTITÉS :
- Si nombre de personnes : adapte les quantités (arrondis réalistes).
- Sinon : quantités originales avec "pour X personnes".

MODE EXPLORATION vs MODE RECETTE :
- Si la demande est ouverte, exploratoire ou non spécifique :
  → propose plusieurs recettes (liste courte)
  → 1 phrase max par recette
  → pas de détails complets
- Si la demande vise une préparation précise :
  → donne une seule recette détaillée

STYLE :
- Commence toujours par une courte phrase naturelle adaptée à la réponse.
- Sois simple, humain et utile.

Contexte : {context}
Question : {question}
Réponse :"""

# """Tu es RAGoût, assistant culinaire expert.
# Réponds UNIQUEMENT à partir des recettes du contexte, sans rien inventer.

# SÉLECTION :
# - Choisis la recette qui correspond le mieux aux ingrédients, au temps et aux préférences mentionnés
# - Si des ingrédients sont donnés, privilégie celles qui en utilisent le maximum
# - Si aucune recette ne correspond, dis-le clairement et liste les recettes proches disponibles

# QUANTITÉS :
# - Si un nombre de personnes est précisé : adapte proportionnellement avec des valeurs réalisables (0.5 œuf → 1 œuf)
# - Sinon : donne les quantités originales avec "Quantités pour X personnes"

# FORMAT :
# **[Nom de la recette]** [Préparation + cuisson]
# Quantités pour X personne(s)
# Ingrédients disponibles : [liste]
# Ingrédients à prévoir : [liste ou "Aucun"]
# Pourquoi cette recette ? [une phrase]
# Instructions : [étapes numérotées]

# Contexte : {context}
# Question : {question}
# Réponse :"""



# """You are an assistant for question-answering tasks.
# Use the following context to answer the question.
# If you don't know the answer, just say that you don't know.
# Use five sentences maximum and keep the answer concise.\n
# Question: {question} \nContext: {context} \nAnswer:"""


llm_prompt = PromptTemplate.from_template(llm_prompt_template)

print(llm_prompt)






input_variables=['context', 'question'] input_types={} partial_variables={} template='Tu es RAGoût, assistant culinaire chaleureux et utile.\n\nTu réponds uniquement à partir des recettes présentes dans le contexte.\nN’invente jamais de recette ni d’ingrédient.\n\nSÉLECTION :\n- Choisis la ou les recettes les plus pertinentes selon les ingrédients, le temps et les préférences.\n- Si des ingrédients sont fournis, privilégie les recettes qui en utilisent le plus.\n- Si aucune recette ne correspond bien, indique-le clairement et propose les recettes les plus proches du contexte.\n- Si on te demande plusieurs idées proposes en plusieurs à l\'utilisateur pour lui laisser le choix \n\nQUANTITÉS :\n- Si un nombre de personnes est précisé, adapte les quantités de façon réaliste (arrondis si nécessaire, ex : 0.5 œuf → 1 œuf).\n- Sinon, garde les quantités originales et indique "pour X personnes".\n\nSTYLE :\n- Sois naturel, chaleureux et conversationnel.\n- Adapte ta réponse à la question. Si l

### Create a stuff documents chain

LangChain provides [Chains](https://python.langchain.com/docs/modules/chains/) for chaining together LLMs with each other or other components for complex applications. You will create a **stuff documents chain** for this application. A stuff documents chain lets you combine all the relevant documents, insert them into the prompt, and pass that prompt to the LLM.

You can create a stuff documents chain using the [LangChain Expression Language (LCEL)](https://python.langchain.com/docs/expression_language).

To learn more about different types of document chains, read LangChain's [chains guide](https://python.langchain.com/docs/modules/chains/document/).

The stuff documents chain for this application retrieves the relevant website data and passes it as the context to an LLM prompt along with the input question.

In [ ]:
# Combine data from documents to readable string format.
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Create stuff documents chain using LCEL.
#
# This is called a chain because you are chaining together different elements
# with the LLM. In the following example, to create the stuff chain, you will
# combine the relevant context from the website data matching the question, the
# LLM model, and the output parser together like a chain using LCEL.
#
# The chain implements the following pipeline:
# 1. Extract the website data relevant to the question from the Chroma
#    vector store and save it to the variable `context`.
# 2. `RunnablePassthrough` option to provide `question` when invoking
#    the chain.
# 3. The `context` and `question` are then passed to the prompt where they
#    are populated in the respective variables.
# 4. This prompt is then passed to the LLM (`gemini-2.0-flash`).
# 5. Output from the LLM is passed through an output parser
#    to structure the model's response.
# rag_chain = (
#     {"context": retriever | format_docs, "question": RunnablePassthrough()}
#     | llm_prompt
#     | llm
#     | StrOutputParser()
# )

from langchain_core.runnables import RunnableLambda

# Retriever avec seuil
def retrieve_with_threshold(question):
    return retrieve_documents_crag(question)

# Chain qui utilise le seuil
rag_chain_avec_seuil = (
    {
        "context": RunnableLambda(retrieve_with_threshold) | format_docs,
        "question": RunnablePassthrough()
    }
    | llm_prompt
    | llm
    | StrOutputParser()
)

### Prompt the model

You can now query the LLM by passing any question to the `invoke()` function of the stuff documents chain you created previously.

#### Partie avec mémoire

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

chat_history = []

# reformulation prompt
reformulation_prompt = PromptTemplate.from_template("""Reformule la question suivante en une question autonome et complète,
en utilisant l'historique de conversation si nécessaire.
Si la question est déjà claire sans contexte, retourne-la telle quelle.
Retourne UNIQUEMENT la question reformulée de manière simple.

Historique:
{chat_history}

Question: {question}

Question reformulée:""")

# answer_prompt
answer_prompt = PromptTemplate.from_template("""Tu es un assistant culinaire. Réponds à la question en te basant sur le contexte fourni.
Si le contexte ne contient pas la réponse, utilise tes connaissances générales.

Historique de conversation:
{chat_history}

Contexte:
{context}

Question: {question}

Réponse:""")

def chat(question):
    history_str = "\n".join([
        f"Humain: {m.content}" if isinstance(m, HumanMessage) else f"Assistant: {m.content}"
        for m in chat_history
    ])

    # reformule
    if chat_history:
        standalone_question = (llm | StrOutputParser()).invoke(
            reformulation_prompt.invoke({"question": question, "chat_history": history_str})
        )
        standalone_question = standalone_question.strip()[:200]
        print(f"Question reformulée : {standalone_question}")
    else:
        standalone_question = question

    # retrieval
    retrieved_docs = retriever.invoke(standalone_question)
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    # réponse
    answer = (llm | StrOutputParser()).invoke(
        answer_prompt.invoke({"question": question, "context": context, "chat_history": history_str})
    )

    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))

    return answer

In [ ]:
# Test — cellule séparée
print(chat("J'ai des pâtes, des tomates et 20 minutes"))
print("---")
print(chat("Et sans les tomates ?"))

Avec des pâtes, des tomates et 20 minutes, je vous suggère de préparer une **Salade de pâtes comme en Italie**.

Elle ne demande que 10 minutes de préparation et 10 minutes de cuisson, ce qui correspond parfaitement à votre temps disponible. Vous aurez besoin de pâtes, de tomates, de mozzarella, d'olives, d'huile d'olive, d'origan, de pesto, d'ail, de sel et de roquette.
---
Question reformulée : Que faire avec des pâtes en 20 minutes sans tomates ?
Dans ce cas, je vous propose des **Tagliatelles au saumon fumé à l'italienne**.

Elles ne demandent que 10 minutes de préparation et 10 minutes de cuisson, soit 20 minutes au total, ce qui correspond parfaitement à votre temps disponible. Vous aurez besoin de tagliatelles, de saumon fumé, d'oignon, de crème liquide, de citron, de persil et de parmesan râpé (facultatif).


In [ ]:
# Part with memory :
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []

def chat(question):
    history_str = "\n".join([
        f"Humain: {m.content}" if isinstance(m, HumanMessage) else f"Assistant: {m.content}"
        for m in chat_history
    ])

    full_question = f"Historique:\n{history_str}\n\nNouvelle question: {question}" if history_str else question

    answer = rag_chain.invoke(full_question)

    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))

    return answer

# Exemple d'utilisation
print(chat("J'ai des tomates et 20 minutes"))
print("---")
print(chat("Et sans les tomates ?"))

NameError: name 'rag_chain' is not defined

In [ ]:
from IPython.display import Markdown

Markdown(rag_chain.invoke("J'ai des tomates, j'ai 20 minutes, et j'adore la cuisine italienne"))

Je ne trouve pas de recette italienne avec des tomates qui puisse être prête en 20 minutes maximum dans le contexte fourni. La recette de "Bruchetta tomate, mozzarella et jambon cru" est italienne et utilise des tomates, mais elle demande 25 minutes au total.

In [ ]:
Markdown(rag_chain.invoke("Donne moi une recette pour faire une galette St Michel"))

Pour faire une galette St Michel, commencez par mélanger 300 g de farine avec 160 g de sucre et 2 œufs, puis incorporez 170 g de beurre ramolli, du sel, un zeste de citron et une pointe de cannelle. Pétrissez le tout pendant 3 minutes pour former une boule, puis placez-la au frais pendant 1 à 2 heures. Étalez ensuite la pâte sur 3 mm d'épaisseur entre deux feuilles de cuisson et remettez-la brièvement au frais. Découpez des galettes à l'aide d'un emporte-pièce, disposez-les sur du papier sulfurisé, puis badigeonnez-les de jaune d'œuf et parsemez-les d'un peu de sucre. Enfin, faites-les cuire au four préchauffé à 160°C (Th.5) pendant 40 à 50 minutes.

Evaluation du RAG

In [ ]:
# Cellule 1
from langchain_google_genai import ChatGoogleGenerativeAI
import json
import pandas as pd

evaluator_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

questions_test = [
    "Peux-tu me donner des idées de plat salés qui utilisent de la courgette et du fromage ?",
    # "J'ai 20 minutes, des tomates et je veux faire une recette italienne",
    # "Comment faire une ratatouille ?",
    # "J'ai que des pommes de terre et des oignons, que puis-je cuisiner ?",
    # "Comment on fait un karkkouf ?",
]

def evaluer_reponse(question, answer, contexts):
    context_str = "\n---\n".join(contexts[:6])
    prompt = f"""Tu es un évaluateur de système RAG. Évalue la réponse suivante sur 4 métriques.
Réponds UNIQUEMENT en JSON valide, sans markdown, sans explication.

Question : {question}
Contextes récupérés : {context_str}
Réponse générée : {answer}

Donne un score entre 0 et 1 pour chaque métrique :
- faithfulness : la réponse est-elle fidèle aux contextes ? (0=invente tout, 1=basée sur les contextes)
- answer_relevancy : la réponse répond-elle à la question ? (0=hors sujet, 1=parfaitement pertinente)
- context_precision : les contextes récupérés sont-ils utiles pour répondre ? (0=inutiles, 1=parfaits)
- noise_sensitivity : le modèle se laisse-t-il influencer par des contextes non pertinents ?
  (0=ignore bien les mauvais docs, 1=utilise tout même ce qui est hors sujet)

Format attendu :
{{"faithfulness": 0.8, "answer_relevancy": 0.9, "context_precision": 0.7, "noise_sensitivity": 0.3, "commentaire": "..."}}"""

    result = evaluator_llm.invoke(prompt)

    # Gemini peut retourner soit un string soit une liste
    content = result.content
    if isinstance(content, list):
        content = " ".join([c.get("text", "") if isinstance(c, dict) else str(c) for c in content])

    text = content.strip().replace("```json", "").replace("```", "").strip()
    return json.loads(text)

In [ ]:
import time

print("🔄 Génération et évaluation en cours...\n")
resultats = []

for i, question in enumerate(questions_test):
    print(f"❓ ({i+1}/{len(questions_test)}) {question}")

    try:
        # Appel 1 : génération
        retrieved_docs = retrieve_documents_crag(question)
        contexts = [doc.page_content for doc in retrieved_docs]
        answer = rag_chain_avec_seuil.invoke(question)
        print(f"💬 {answer}")

        # Pause entre les 2 appels
        time.sleep(30)

        # Appel 2 : évaluation
        scores = evaluer_reponse(question, answer, contexts)
        scores["question"] = question
        scores["answer"] = answer
        resultats.append(scores)
        print(f"📊 {scores}")

    except Exception as e:
        print(f"⚠️ Erreur sur cette question, on attend 60s et on continue... ({e})")
        time.sleep(60)
        continue

    # Pause entre chaque question
    print(f"⏳ Pause 30s avant la question suivante...\n")
    time.sleep(30)

print(f"\n✅ Évaluation terminée ! ({len(resultats)}/{len(questions_test)} questions traitées)")

🔄 Génération et évaluation en cours...

❓ (1/1) Peux-tu me donner des idées de plat salés qui utilisent de la courgette et du fromage ?
⚠️ Erreur sur cette question, on attend 60s et on continue... (Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 40.743533915s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFa

KeyboardInterrupt: 

In [ ]:
# Cellule 3 — Résultats
df = pd.DataFrame(resultats)

print("=" * 60)
print("ÉVALUATION RAG — SCORES DE RÉFÉRENCE")
print("=" * 60)
print(f"\nFaithfulness      (fidélité)   : {df['faithfulness'].mean():.3f} / 1.0")
print(f"Answer Relevancy  (pertinence) : {df['answer_relevancy'].mean():.3f} / 1.0")
print(f"Context Precision (contexte)   : {df['context_precision'].mean():.3f} / 1.0")
print(f"\nMoyenne globale                : {df[['faithfulness','answer_relevancy','context_precision']].mean().mean():.3f} / 1.0")
print("\n📋 Détail par question :")
print(df[['question','faithfulness','answer_relevancy','context_precision','commentaire']].to_string(index=False))

ÉVALUATION RAG — SCORES DE RÉFÉRENCE

Faithfulness      (fidélité)   : 0.936 / 1.0
Answer Relevancy  (pertinence) : 0.940 / 1.0
Context Precision (contexte)   : 0.731 / 1.0

Moyenne globale                : 0.869 / 1.0

📋 Détail par question :
                                                              question  faithfulness  answer_relevancy  context_precision                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          commentaire
                                   Comment faire une quiche lorraine ?          1.00              1.00               0.90                            

# Conclusion

That's it. You have successfully created an LLM application that answers questions using data from a website with the help of Gemini, LangChain, and Chroma.